In [7]:
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LogisticRegression, LinearRegression
# Automatically finds the project directory and adds it to the Python environment path
sys.path.append(os.path.abspath(".."))

# Import custom modules from your uploaded codebase
from src.encoders import OneHotEncoder
from src.imputers import SimpleImputer as BaseSimpleImputer
from src.scalers import StandardScaler as BaseStandardScaler


# --- 1. Scikit-Learn Compatibility Adapters ---
class SklearnSimpleImputer(BaseSimpleImputer, BaseEstimator, TransformerMixin):

    def __init__(self, strategy="mean"):
        super().__init__(strategy=strategy)

    # [BUGFIX PATCH]: Intercepts the base fit(), calculates the stats, and forces it to return 'self'
    def fit(self, X, y=None):
        super().fit(X, y)
        return self


class SklearnStandardScaler(
    BaseStandardScaler, BaseEstimator, TransformerMixin
):

    def __init__(self):
        super().__init__()


# --- 2. Custom Age Missing-Indicator Step ---
class AgeMissingIndicator(BaseEstimator, TransformerMixin):
    """Safely extracts the binary missing-value indicator for 'age' before imputation."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df["age_missing"] = X_df["age"].isna().astype(int)
        return X_df


# --- 3. Load & Split Titanic Dataset ---
titanic = sns.load_dataset("titanic")
features = ["age", "fare", "sibsp", "parch", "sex", "embarked"]

X = titanic[features].copy()
y = titanic["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 4. Define Preprocessing Branches via ColumnTransformer ---
numeric_cols = ["age", "fare", "sibsp", "parch"]
categorical_cols = ["embarked", "sex"]
indicator_cols = ["age_missing"]

# Numeric Branch: Imputes missing age with median -> Standardises all 4 numeric features
num_pipeline = Pipeline(
    [
        ("imputer", SklearnSimpleImputer(strategy="median")),
        ("scaler", SklearnStandardScaler()),
    ]
)

# Categorical Branch: Imputes missing embarked with mode -> One-Hot Encodes (dropping first)
cat_pipeline = Pipeline(
    [
        ("imputer", SklearnSimpleImputer(strategy="mode")),
        ("encoder", OneHotEncoder(drop="first")),
    ]
)

# Assemble multi-branch preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numeric_cols),
        ("cat", cat_pipeline, categorical_cols),
        ("ind", "passthrough", indicator_cols),
    ]
)

# --- 5. Assemble Master Pipeline ---
titanic_master_pipe = Pipeline(
    [
        ("indicator_gen", AgeMissingIndicator()),
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
    ]
)

# --- 6. Execute 5-Fold Cross-Validation ---
cv_scores = cross_val_score(
    titanic_master_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

print("=== TITANIC MASTER PIPELINE EVALUATION ===")
print(f"5-Fold CV Accuracy Scores: {cv_scores}")
print(f"Mean CV Accuracy:          {cv_scores.mean() * 100:.2f}%")
print(f"Standard Deviation:        ±{cv_scores.std() * 100:.2f}%")

=== TITANIC MASTER PIPELINE EVALUATION ===
5-Fold CV Accuracy Scores: [0.8041958  0.81118881 0.78873239 0.73239437 0.8028169 ]
Mean CV Accuracy:          78.79%
Standard Deviation:        ±2.87%


| # | Action Taken in Codebase             | What the Leakage Would Have Been                                                                                                                                                                                                                                                                | How It Was Avoided                                                                                                                                                                                                                                                                                |
| - | ------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1 | **Out-of-Fold Target Encoding Loop** | If a category is replaced by the average target value of all rows in that category globally, the feature effectively memorizes its own label ((y_i)), creating severe target leakage and leading to unrealistically high validation performance followed by poor generalization on unseen data. | In `encoders.py`, `TargetEncoder.fit_transform()` implements an internal K-Fold loop. For each training row, the category mean is computed exclusively from the other (K-1) folds, ensuring that the row's own target value is never used when generating its encoded feature.                    |
| 2 | **Imputer Parameter Isolation**      | Computing imputation statistics (e.g., median age or most frequent embarked value) on the full dataset before cross-validation leaks information from the validation fold into the training process. This exposes the model to the distribution of unseen data.                                 | All median and mode calculations are encapsulated within the `fit()` methods of the transformers. During `cross_val_score`, Scikit-Learn calls `fit()` only on the training portion of each fold, ensuring that validation data remains completely unseen when imputation parameters are learned. |
| 3 | **Downstream Scaling Execution**     | Standardizing features using a global mean ((\mu)) and standard deviation ((\sigma)) computed from the entire dataset leaks information about the overall scale and variance of the validation data into the training process.                                                                  | `StandardScaler` is placed inside a `Pipeline`. During the pipeline's `fit()` stage, the scaler computes (\mu) and (\sigma) exclusively from the training fold. The learned parameters are then applied unchanged to the validation fold during transformation.                                   |

## Summary

The codebase prevents data leakage by ensuring that every data-dependent transformation—target encoding, imputation, and scaling—learns its parameters exclusively from the training partition of each cross-validation fold. Validation data is never used during parameter estimation, preserving the integrity of model evaluation and providing a realistic estimate of generalization performance.


In [8]:
# --- Custom Winsorisation Transformer ---
class Winsorizer(BaseEstimator, TransformerMixin):
    """Clips extreme tail values at empirical lower and upper training quantiles."""

    def __init__(self, lower_quantile=0.01, upper_quantile=0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X_arr = np.asarray(X, dtype=float)
        self.lower_bounds_ = np.percentile(
            X_arr, self.lower_quantile * 100, axis=0
        )
        self.upper_bounds_ = np.percentile(
            X_arr, self.upper_quantile * 100, axis=0
        )
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return np.clip(X_arr, self.lower_bounds_, self.upper_bounds_)


# --- 1. Load California Housing Dataset ---
X_cal, y_cal = fetch_california_housing(return_X_y=True)
X_cal_train, X_cal_test, y_cal_train, y_cal_test = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)

# --- 2. Pipeline A: Untreated Standard Scaling ---
pipe_untreated = Pipeline(
    [("scaler", SklearnStandardScaler()), ("ols", LinearRegression())]
)

# --- 3. Pipeline B: Winsorised MedInc (Col 0) + Standard Scaling ---
winsor_step = ColumnTransformer(
    transformers=[
        ("winsor_medinc", Winsorizer(0.01, 0.99), [0]),  # Column 0 is MedInc
        ("pass_rest", "passthrough", slice(1, 8)),  # Pass features 1 through 7
    ]
)

pipe_winsorised = Pipeline(
    [
        ("winsor", winsor_step),
        ("scaler", SklearnStandardScaler()),
        ("ols", LinearRegression()),
    ]
)

# --- 4. Fit & Extract Coefficients ---
pipe_untreated.fit(X_cal_train, y_cal_train)
pipe_winsorised.fit(X_cal_train, y_cal_train)

# ColumnTransformer keeps MedInc at Index 0 in both pipelines
coef_untreated = pipe_untreated.named_steps["ols"].coef_[0]
coef_winsorised = pipe_winsorised.named_steps["ols"].coef_[0]

# --- 5. Report 5-Fold Training RMSE ---
rmse_untreated = -cross_val_score(
    pipe_untreated,
    X_cal_train,
    y_cal_train,
    scoring="neg_root_mean_squared_error",
    cv=5,
).mean()

rmse_winsorised = -cross_val_score(
    pipe_winsorised,
    X_cal_train,
    y_cal_train,
    scoring="neg_root_mean_squared_error",
    cv=5,
).mean()

print("=== WINSORISATION EXPERIMENT RESULTS ===")
print(
    f"Untreated MedInc Coefficient:  {coef_untreated:.4f}  |  CV RMSE: {rmse_untreated:.4f}"
)
print(
    f"Winsorised MedInc Coefficient: {coef_winsorised:.4f}  |  CV RMSE: {rmse_winsorised:.4f}"
)

=== WINSORISATION EXPERIMENT RESULTS ===
Untreated MedInc Coefficient:  0.8544  |  CV RMSE: 0.7205
Winsorised MedInc Coefficient: 0.9022  |  CV RMSE: 0.7071


## Why Winsorisation Improves Model Generalisation

Ordinary Least Squares (OLS) linear regression operates on a quadratic cost function, attempting to minimize the sum of squared residuals:

$$
\mathcal{L}(\beta) = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

Because the error term is squared, extreme outliers exert a disproportionate influence on the optimization process. A small number of observations with very large residuals can dominate the loss function and pull the regression coefficients away from values that best represent the majority of the data.

In the raw California Housing dataset, the `MedInc` feature contains severe upper-tail anomalies, including census blocks with median income values reaching the dataset cap of approximately (15.0001). If these observations are left untreated, the OLS optimizer may distort the primary slope coefficient ((\beta_1)) in an attempt to reduce the large squared penalties associated with these extreme leverage points. Consequently, the fitted model becomes less representative of the underlying population trend.

Extreme anomalies also affect feature standardization. `StandardScaler` computes the standardized feature as

$$
X_{\text{scaled}} = \frac{X - \mu}{\sigma},
$$

where (\mu) is the sample mean and (\sigma) is the sample standard deviation.

A handful of unusually large values can substantially inflate (\sigma). When all observations are divided by this inflated standard deviation, the majority of typical middle-income households become compressed into a narrow range around zero, reducing the effective resolution of the feature.

Winsorisation addresses this issue by capping extreme observations at predefined percentile thresholds. By winsorising at the 1st and 99th empirical percentiles, values beyond these cutoffs are replaced with the corresponding boundary values. This reduces the influence of extreme tails while preserving all observations in the dataset.

As a result:

* The sample standard deviation becomes more representative of the bulk of the distribution.
* The scaling process retains greater separation among typical observations.
* The OLS coefficients become less sensitive to extreme leverage points.
* The model is less likely to overfit rare tail anomalies.
* Cross-validated RMSE often improves because the learned relationship better reflects the underlying population rather than a small number of extreme observations.

Therefore, winsorisation enhances model generalisation by reducing the influence of outliers on both the optimization procedure and the feature-scaling process, leading to more stable parameter estimates and improved predictive performance on unseen data.
